# Балансировка классов и улучшение разделимости

Методы борьбы с несбалансированными данными для задач классификации.

## Содержание

1. **Проблема несбалансированных классов** — почему это проблема, наглядная демонстрация.
2. **Метрики качества для несбалансированных данных** — precision, recall, F1, ROC-AUC, PR-AUC.
3. **Базовый пайплайн** — обучение модели без балансировки как точка отсчёта.
4. **Undersampling (снижение частоты мажоритарного класса)**:
   - RandomUnderSampler
   - TomekLinks
   - ClusterCentroids
   - NearMiss
5. **Oversampling (повышение частоты миноритарного класса)**:
   - RandomOverSampler
   - SMOTE
   - BorderlineSMOTE
   - ADASYN
6. **Комбинированные методы**:
   - SMOTE + TomekLinks
   - SMOTE + ENN
7. **Улучшение разделимости классов**:
   - Преобразование признаков (Box-Cox, Yeo-Johnson)
   - Создание новых признаков
   - Отбор признаков для разделимости
8. **Cost-sensitive learning** — использование `class_weight` в sklearn.
9. **Сравнение методов** — сводная таблица метрик и визуализация.
10. **Рекомендации** — когда что применять.

In [ ]:
!uv pip install --system imbalanced-learn pandas numpy matplotlib seaborn scikit-learn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve,
    f1_score, recall_score, precision_score
)
from sklearn.preprocessing import StandardScaler, PowerTransformer, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif

sns.set_style("darkgrid")
pd.options.display.float_format = "{:.4f}".format
np.random.seed(42)

---
## 1. Проблема несбалансированных классов

Если один из классов представлен значительно меньшим количеством объектов, чем другой, модель склонна «игнорировать» миноритарный класс — ей «выгодно» предсказывать всегда мажоритарный класс, чтобы получить высокий `accuracy`.

**Пример:** детекция мошеннических транзакций — 99.9% легитимных, 0.1% мошеннических. Модель, которая всегда предсказывает «легитимно», получит accuracy 99.9%, но не обнаружит ни одного мошенничества.

Сгенерируем синтетический датасет с дисбалансом для демонстрации.

In [ ]:
def make_imbalanced_dataset(
    n_samples=5000, n_features=8, n_informative=5, n_redundant=2,
    weights=None, flip_y=0.05, random_state=42
):
    """Generate a binary classification dataset with controlled imbalance."""
    if weights is None:
        weights = [0.9, 0.1]
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=n_informative,
        n_redundant=n_redundant,
        n_clusters_per_class=1,
        weights=weights,
        flip_y=flip_y,
        class_sep=0.8,
        random_state=random_state,
    )
    return X, y

X, y = make_imbalanced_dataset()

df = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
df["target"] = y

print(f"Всего объектов: {len(df)}")
print(f"Миноритарный класс (1): {y.sum()} ({y.mean()*100:.1f}%)")
print(f"Мажоритарный класс (0): {(1-y).sum()} {(1-y.mean())*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["target"].value_counts().plot(kind="bar", ax=axes[0], color=["steelblue", "coral"])
axes[0].set_title("Распределение классов")
axes[0].set_xticklabels(["Мажоритарный (0)", "Миноритарный (1)"], rotation=0)

from pandas.plotting import parallel_coordinates
df_sample = pd.concat([df[df["target"]==0].sample(50),
                       df[df["target"]==1].sample(50)])
df_sample["target"] = df_sample["target"].map({0: "Мажоритарный", 1: "Миноритарный"})
parallel_coordinates(df_sample, class_column="target", ax=axes[1], color=["steelblue", "coral"])
axes[1].set_title("Признаки объектов двух классов")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

---
## 2. Метрики качества для несбалансированных данных

При дисбалансе **accuracy (доля правильных ответов) обманчива**. Нужны метрики, чувствительные к ошибкам на миноритарном классе:

| Метрика | Формула | Что показывает |
|---|---|---|
| **Precision** (точность) | TP / (TP + FP) | Доля верно предсказанных положительных среди всех, кто был отнесён к положительным |
| **Recall** (полнота) | TP / (TP + FN) | Доля найденных положительных объектов от всех реальных положительных |
| **F1-score** | 2 · P · R / (P + R) | Гармоническое среднее precision и recall |
| **ROC-AUC** | площадь под TPR/FPR | Способность модели ранжировать — разделять классы (при сбалансированных классах) |
| **PR-AUC** (average precision) | площадь под Precision/Recall | Лучше ROC-AUC при сильном дисбалансе |

**Правило:** при дисбалансе > 1:10 смотрите в первую очередь на **PR-AUC** и **F1 по миноритарному классу**.

In [ ]:
# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Train: {y_train.sum()} positive / {len(y_train)} total ({y_train.mean()*100:.1f}%)")
print(f"Test:  {y_test.sum()} positive / {len(y_test)} total ({y_test.mean()*100:.1f}%)")

---
## 3. Базовый пайплайн — модель без балансировки

In [ ]:
def evaluate_model(model, X_test, y_test, model_name="Model"):
    """Evaluate and print metrics for a classifier."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results = {
        "Accuracy": model.score(X_test, y_test),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "PR-AUC": average_precision_score(y_test, y_proba),
    }

    print(f"\n{'='*50}")
    print(f"{model_name}")
    print(f"{'='*50}")
    for k, v in results.items():
        print(f"  {k:12s}: {v:.4f}")
    print(f"\n{classification_report(y_test, y_pred, target_names=['Majority', 'Minority'])}")

    return results


# Baseline: RandomForest без балансировки
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

baseline_model = RandomForestClassifier(n_estimators=200, random_state=42)
baseline_model.fit(X_train_scaled, y_train)

baseline_results = evaluate_model(baseline_model, X_test_scaled, y_test, model_name="BASELINE — без балансировки")

# Confusion matrix
cm = confusion_matrix(y_test, baseline_model.predict(X_test_scaled))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Majority", "Pred Minority"],
            yticklabels=["True Majority", "True Minority"])
plt.title("Confusion Matrix — Baseline")
plt.show()

---
## 4. Undersampling — снижение частоты мажоритарного класса

Идея: уменьшить количество объектов мажоритарного класса, чтобы уравнять частоты.

**Плюсы:** простой, быстрый.
**Минусы:** теряем данные — можно выбросить полезные примеры.

Пакет `imbalanced-learn` (`imblearn`) предоставляет готовые реализации.

In [ ]:
from imblearn.under_sampling import (
    RandomUnderSampler,
    TomekLinks,
    ClusterCentroids,
    NearMiss,
)

# Helper: train + evaluate with a sampler
def train_with_sampler(sampler, X_train, y_train, X_test, y_test, name="Sampler"):
    X_res, y_res = sampler.fit_resample(X_train, y_train)
    unique, counts = np.unique(y_res, return_counts=True)
    print(f"{name}: после ресемплинга {dict(zip(unique, counts))}")

    model = RandomForestClassifier(n_estimators=200, random_state=42)
    model.fit(X_res, y_res)
    return evaluate_model(model, X_test, y_test, model_name=name)

In [ ]:
# --- RandomUnderSampler ---
rus_results = train_with_sampler(
    RandomUnderSampler(random_state=42),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name="RandomUnderSampler"
)

In [ ]:
# --- TomekLinks ---
# Удаляет «сомнительные» объекты мажоритарного класса на границе классов.
# Объекты из разных классов, которые являются ближайшими соседями, образуют «Tomek link».
# Мажоритарный объект из такой пары удаляется — это очищает границу.

tl_results = train_with_sampler(
    TomekLinks(),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name="TomekLinks"
)

In [ ]:
# --- ClusterCentroids ---
# Заменяет объекты мажоритарного класса на K их центроидов (k-средних).
# Не удаляет, а синтезирует новые центроиды вместо исходных объектов.

cc_results = train_with_sampler(
    ClusterCentroids(random_state=42),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name="ClusterCentroids"
)

In [ ]:
# --- NearMiss ---
# Отбирает объекты мажоритарного класса, ближайшие к миноритарному (по расстоянию).
# Варианты:
#   version=1 — среднее расстояние до K ближайших миноритарных соседей
#   version=2 — среднее расстояние до K farthest миноритарных соседей
#   version=3 — для каждого мажоритарного объекта сохранить, если он среди N ближайших соседей каждого миноритарного

nm1_results = train_with_sampler(
    NearMiss(version=1),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name="NearMiss-v1"
)

nm3_results = train_with_sampler(
    NearMiss(version=3),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name="NearMiss-v3"
)

---
## 5. Oversampling — повышение частоты миноритарного класса

Идея: создать новые объекты миноритарного класса, чтобы уравнять частоты.

**Плюсы:** не теряем данные мажоритарного класса.
**Минусы:** может привести к переобучению, если синтезированные объекты повторяют реальные.

In [ ]:
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.over_sampling import BorderlineSMOTE

In [ ]:
# --- RandomOverSampler ---
# Просто дублирует случайные объекты миноритарного класса.
# Самый быстрый, но最容易 к переобучению.

ros_results = train_with_sampler(
    RandomOverSampler(random_state=42),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name="RandomOverSampler"
)

In [ ]:
# --- SMOTE ---
# Synthetic Minority Oversampling Technique.
# Создаёт новые синтетические объекты между существующими миноритарными объектами и их K соседями.
# Для каждого миноритарного объекта:
#   1. Выбрать случайного соседа среди K ближайших
#   2. Создать новый объект в случайной точке отрезка между ними

smote_results = train_with_sampler(
    SMOTE(random_state=42),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name="SMOTE"
)

In [ ]:
# --- BorderlineSMOTE ---
# Вариант SMOTE: создаёт объекты ТОЛЬКО для миноритарных объектов на границе классов.
# Классифицирует миноритарные объекты как «безопасные», «граничные», «шумовые».
# Oversampling применяется только к «граничным» — они самые полезные для улучшения разделимости.

bsmote_results = train_with_sampler(
    BorderlineSMOTE(random_state=42),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name="BorderlineSMOTE"
)

In [ ]:
# --- ADASYN ---
# Adaptive Synthetic Sampling.
# Похож на SMOTE, но количество синтезируемых объектов пропорционально сложности:
# чем больше мажоритарных соседей у миноритарного объекта, тем больше новых точек создаётся вокруг него.
# То есть больше объектов создаётся там, где классы перемешаны сильнее.

adasyn_results = train_with_sampler(
    ADASYN(random_state=42),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name="ADASYN"
)

---
## 6. Комбинированные методы

Сначала oversampling (SMOTE), затем undersampling для очистки границы.

In [ ]:
from imblearn.combine import SMOTETomek, SMOTEENN

In [ ]:
# --- SMOTE + TomekLinks ---
# 1. SMOTE создаёт синтетические объекты миноритарного класса
# 2. TomekLinks удаляет «сомнительные» объекты с обеих сторон границы
# Итог: больше миноритарных + чище граница между классами

smt_results = train_with_sampler(
    SMOTETomek(random_state=42),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name="SMOTE+TomekLinks"
)

In [ ]:
# --- SMOTE + ENN ---
# 1. SMOTE создаёт синтетические объекты
# 2. Edited Nearest Neighbours удаляет объекты, чей класс не совпадает с классом большинства соседей
# Более агрессивная очистка, чем TomekLinks

sme_results = train_with_sampler(
    SMOTEENN(random_state=42),
    X_train_scaled, y_train, X_test_scaled, y_test,
    name="SMOTE+ENN"
)

---
## 7. Улучшение разделимости классов

Помимо балансировки, можно улучшить саму структуру признаков, чтобы классы было легче разделить.

### 7.1 Преобразование признаков

- **Box-Cox** — приводит распределение к более нормальному, но работает только с положительными значениями.
- **Yeo-Johnson** — обобщение Box-Cox, работает с любыми значениями.
- **Логарифмирование** — сжимает длинный хвост, полезно при сильно скошенных распределениях.

Нормализация распределений помогает моделям (особенно линейным) лучше разделять классы.

In [ ]:
# Генерируем данные со скошенными распределениями
np.random.seed(42)
n_skew = 2000

f_skewed = np.random.exponential(scale=2, size=n_skew)  # сильно скошенное распределение
f_normal = np.random.normal(loc=0, scale=1, size=n_skew)
y_skew = (f_normal + 0.3 * f_skewed + np.random.normal(0, 0.5, n_skew) > 2).astype(int)

X_skew_train, X_skew_test, y_skew_train, y_skew_test = train_test_split(
    np.column_stack([f_skewed, f_normal]), y_skew, test_size=0.3, random_state=42
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(f_skewed, bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Исходное распределение (Exponential)")

# Log transform
f_log = np.log1p(f_skewed)
axes[1].hist(f_log, bins=50, color="coral", edgecolor="white")
axes[1].set_title("После log1p")

# Yeo-Johnson
pt = PowerTransformer(method="yeo-johnson")
f_yj = pt.fit_transform(f_skewed.reshape(-1, 1)).ravel()
axes[2].hist(f_yj, bins=50, color="green", edgecolor="white", alpha=0.7)
axes[2].set_title("После Yeo-Johnson")
plt.tight_layout()
plt.show()

# Сравнение качества LogisticRegression
lr_raw = LogisticRegression().fit(X_skew_train, y_skew_train)
print(f"ROC-AUC на сырых признаках: {roc_auc_score(y_skew_test, lr_raw.predict_proba(X_skew_test)[:,1]):.4f}")

# Apply Yeo-Johnson
pt_full = PowerTransformer(method="yeo-johnson")
X_skew_train_t = pt_full.fit_transform(X_skew_train)
X_skew_test_t = pt_full.transform(X_skew_test)
lr_t = LogisticRegression().fit(X_skew_train_t, y_skew_train)
print(f"ROC-AUC после Yeo-Johnson:    {roc_auc_score(y_skew_test, lr_t.predict_proba(X_skew_test_t)[:,1]):.4f}")

### 7.2 Создание новых признаков

Полиномиальные признаки (`PolynomialFeatures`) и ручное создание комбинаций могут помочь, если разделяющая граница нелинейна.

In [ ]:
# Демонстрация: нелинейная разделяющая граница
np.random.seed(42)
n_circ = 1000
t = np.random.uniform(0, 2 * np.pi, n_circ)
r = np.random.uniform(0, 1, n_circ)

X_circ = np.column_stack([
    r * np.cos(t) + np.random.normal(0, 0.1, n_circ),
    r * np.sin(t) + np.random.normal(0, 0.1, n_circ),
])
y_circ = np.random.binomial(1, 0.5, n_circ)

# Добавим внешний класс для наглядности
r2 = np.random.uniform(1.5, 2.5, n_circ // 2)
t2 = np.random.uniform(0, 2 * np.pi, n_circ // 2)
X_outer = np.column_stack([r2 * np.cos(t2), r2 * np.sin(t2)])
y_outer = np.ones(n_circ // 2)

X_circ = np.vstack([X_circ, X_outer])
y_circ = np.hstack([y_circ, y_outer])

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_circ, y_circ, test_size=0.3, random_state=42
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(Xc_train[yc_train==0, 0], Xc_train[yc_train==0, 1],
            c="steelblue", alpha=0.5, label="Class 0", s=10)
axes[0].scatter(Xc_train[yc_train==1, 0], Xc_train[yc_train==1, 1],
            c="coral", alpha=0.5, label="Class 1", s=10)
axes[0].set_title("Исходные признаки")
axes[0].legend()

poly = PolynomialFeatures(degree=2, include_bias=False)
Xc_train_poly = poly.fit_transform(Xc_train)
Xc_test_poly = poly.transform(Xc_test)

lr_circ = LogisticRegression().fit(Xc_train, yc_train)
lr_circ_poly = LogisticRegression().fit(Xc_train_poly, yc_train)

print(f"LogReg на сырых признаках:    ROC-AUC = {roc_auc_score(yc_test, lr_circ.predict_proba(Xc_test)[:,1]):.4f}")
print(f"LogReg с полином. признаками: ROC-AUC = {roc_auc_score(yc_test, lr_circ_poly.predict_proba(Xc_test_poly)[:,1]):.4f}")

# Визуализация предсказаний
xx, yy = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-3, 3, 100))
Z = lr_circ_poly.predict_proba(poly.transform(
    np.c_[xx.ravel(), yy.ravel()]
))[:, 1].reshape(xx.shape)

axes[1].contourf(xx, yy, Z, alpha=0.3, cmap="RdBu_r", levels=20)
axes[1].scatter(Xc_test[yc_test==0, 0], Xc_test[yc_test==0, 1],
            c="steelblue", alpha=0.5, s=10)
axes[1].scatter(Xc_test[yc_test==1, 0], Xc_test[yc_test==1, 1],
            c="coral", alpha=0.5, s=10)
axes[1].set_title("LogReg с полиномиальными признаками")
plt.tight_layout()
plt.show()

### 7.3 Отбор признаков для улучшения разделимости

Удаление шумовых признаков может заметно улучшить разделимость классов. `SelectKBest` с `f_classif` (ANOVA F-тест) выбирает признаки с наибольшей разделяющей способностью.

Это особенно полезно перед SMOTE: меньше шума — более качественные синтетические объекты.

In [ ]:
# Добавим шумовые признаки к нашему датасету
np.random.seed(42)
noise_cols = np.random.randn(X_train.shape[0], 20)
X_train_noisy = np.hstack([X_train_scaled, noise_cols])
X_test_noisy = np.hstack([X_test_scaled, np.random.randn(X_test.shape[0], 20)])

print(f"Всего признаков (с шумовыми): {X_train_noisy.shape[1]}")

# Baseline с шумом
rf_noisy = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train_noisy, y_train)
print(f"\nRF на всех признаках (с шумом):")
print(f"  ROC-AUC = {roc_auc_score(y_test, rf_noisy.predict_proba(X_test_noisy)[:,1]):.4f}")
print(f"  PR-AUC  = {average_precision_score(y_test, rf_noisy.predict_proba(X_test_noisy)[:,1]):.4f}")

# Отбор 10 лучших признаков
selector = SelectKBest(f_classif, k=10)
X_train_selected = selector.fit_transform(X_train_noisy, y_train)
X_test_selected = selector.transform(X_test_noisy)

rf_sel = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train_selected, y_train)
print(f"\nRF после SelectKBest (k=10):")
print(f"  ROC-AUC = {roc_auc_score(y_test, rf_sel.predict_proba(X_test_selected)[:,1]):.4f}")
print(f"  PR-AUC  = {average_precision_score(y_test, rf_sel.predict_proba(X_test_selected)[:,1]):.4f}")

# SMOTE после отбора признаков
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train_selected, y_train)
rf_smote_sel = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_res, y_res)
print(f"\nSelectKBest + SMOTE + RF:")
print(f"  ROC-AUC = {roc_auc_score(y_test, rf_smote_sel.predict_proba(X_test_selected)[:,1]):.4f}")
print(f"  PR-AUC  = {average_precision_score(y_test, rf_smote_sel.predict_proba(X_test_selected)[:,1]):.4f}")

---
## 8. Cost-sensitive learning — учёт стоимости ошибок

Вместо изменения данных можно сказать модели: «ошибка на миноритарном классе важнее».

В sklearn это параметр **`class_weight`**:
- `'balanced'` —权重 обратно пропорциональны частотам классов
- `{0: w0, 1: w1}` — ручное задание весов

Работает для: LogisticRegression, SVM, RandomForest, GradientBoosting и др.

In [ ]:
models_cw = {
    "LogReg (no weight)": LogisticRegression(random_state=42),
    "LogReg (balanced)":   LogisticRegression(class_weight="balanced", random_state=42),
    "RF (no weight)":      RandomForestClassifier(n_estimators=200, random_state=42),
    "RF (balanced)":       RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "RF (balanced_subsample)": RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample", random_state=42),
}

cw_results = {}
for name, model in models_cw.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    cw_results[name] = {
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "PR-AUC": average_precision_score(y_test, y_proba),
    }
    print(f"{name:30s} | F1={cw_results[name]['F1']:.4f} | ROC-AUC={cw_results[name]['ROC-AUC']:.4f} | PR-AUC={cw_results[name]['PR-AUC']:.4f}")

---
## 9. Сравнение всех методов

Соберём все результаты в одну таблицу и визуализируем.

In [ ]:
all_results = {
    "Baseline": baseline_results,
    "RandomUnderSampler": rus_results,
    "TomekLinks": tl_results,
    "ClusterCentroids": cc_results,
    "NearMiss-v1": nm1_results,
    "NearMiss-v3": nm3_results,
    "RandomOverSampler": ros_results,
    "SMOTE": smote_results,
    "BorderlineSMOTE": bsmote_results,
    "ADASYN": adasyn_results,
    "SMOTE+Tomek": smt_results,
    "SMOTE+ENN": sme_results,
}

comparison_df = pd.DataFrame(all_results).T
comparison_df = comparison_df.sort_values("F1", ascending=False)

print("\n" + "="*90)
print("Сравнение методов балансировки (sorted by F1)")
print("="*90)
comparison_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = ["F1", "ROC-AUC", "PR-AUC"]
colors = ["steelblue" if m != "F1" else "coral" for m in metrics]

for ax, metric, color in zip(axes, metrics, colors):
    sorted_data = comparison_df[metric].sort_values(ascending=True)
    ax.barh(range(len(sorted_data)), sorted_data.values, color=color, alpha=0.7)
    ax.set_yticks(range(len(sorted_data)))
    ax.set_yticklabels(sorted_data.index, fontsize=9)
    ax.set_xlabel(metric)
    ax.set_title(f"{metric} по методам")
    for i, v in enumerate(sorted_data.values):
        ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

---
## 10. Рекомендации

### Когда что применять:

| Ситуация | Рекомендация |
|---|---|
| Данных **очень много**, дисбаланс умеренный | **RandomUnderSampler** — быстро и просто |
| Данных **мало**, дисбаланс сильный | **SMOTE** или **BorderlineSMOTE** |
| Граница классов размыта, много пересечения | **SMOTE + TomekLinks** или **SMOTE + ENN** |
| Очень сильный дисбаланс (> 1:100) | **ADASYN** или комбинация undersampling + oversampling |
| Нельзя менять распределение данных | **class_weight='balanced'** — без изменения данных |
| Модель — дерево/ансамбль | **class_weight** + можно лёгкий oversampling |
| Модель — линейная (LogReg, SVM) | **SMOTE** + преобразование признаков (PowerTransformer) |

### Общие принципы:

1. **Не оценивайте по accuracy** — используйте F1, PR-AUC, Recall для миноритарного класса.
2. **Отбор признаков перед SMOTE** — шумовые признаки ухудшают качество синтетических объектов.
3. **Масштабирование перед SMOTE** — SMOTE использует расстояния, признаки должны быть в одном масштабе.
4. **Разделяйте train/test до ресемплинга** — никогда не делайте SMOTE на всём датасете до split, иначе данные протекут из test в train.
5. **Пробуйте несколько методов** — нет единого лучшего, всё зависит от данных.
6. **Комбинируйте** — преобразование признаков + отбор + SMOTE + correct model часто даёт лучший результат, чем каждый метод по отдельности.

---
## Ссылки

1. [imbalanced-learn documentation](https://imbalanced-learn.org/stable/)
2. [SMOTE — оригинальная статья (Chawla et al., 2002)](https://arxiv.org/abs/1106.1813)
3. [Borderline-SMOTE (Han et al., 2005)](https://www.sciencedirect.com/science/article/abs/pii/S0950705105000773)
4. [ADASYN (He et al., 2008)](https://ieeexplore.ieee.org/document/4633969)
5. [NearMiss — статья (Mani & Zhang, 2003)](https://www.semanticscholar.org/paper/KNN-approach-to-unbalanced-data-distributions%3A-a-Mani-Zhang/)
6. Scikit-learn: [class_weight](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html), [PowerTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html)
7. [Несбалансированные выборки — habr](https://habr.com/ru/articles/686758/)